# Agentes de IA con LangChain y Gemini — Versión completa (RAG + Memoria)

## ¿Qué añade este notebook respecto al anterior?

Este notebook contiene todo lo del notebook anterior (modelo estático, dinámico, tools, stream, structured output, mensajes) y añade dos capacidades nuevas fundamentales:

- **RAG** (Retrieval-Augmented Generation): el agente puede buscar en documentos propios antes de responder
- **Memoria**: el agente recuerda lo que se ha dicho antes en la misma conversación

Estas dos capacidades son las que transforman un chatbot genérico en un asistente útil para un caso de uso real.

## Paso 0 — Instalación y carga de la clave

La API Key es la contraseña que identifica nuestras peticiones a Google Gemini. Se guarda en Secrets de Colab para no exponerla en el código.

In [1]:
import os
from dotenv import load_dotenv

# Lee el archivo .env si existe
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# En Google Colab usa: from google.colab import userdata; GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
# En local con .env usa: GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
print("✅ API Key cargada")

✅ API Key cargada


In [2]:
# Instala si no lo tienes ya
# !pip install langchain-core langchain

### Importaciones

Cargamos todos los componentes que usaremos a lo largo del notebook.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langchain.agents import create_agent

---

## Sección 1 — Modelo estático

El agente más básico: siempre usa el mismo modelo, con los mismos parámetros.

- `MODEL = "gemini-2.5-flash-lite"` → la versión más ligera y rápida de Gemini
- `temperature=0.5` → equilibrio entre respuestas precisas y variadas
- `tools=None` → sin herramientas; solo conversación

In [4]:
MODEL = "gemini-2.5-flash-lite"

In [5]:
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=GOOGLE_API_KEY)

In [6]:
agent = create_agent(llm, tools=None)

---

## Sección 2 — Modelo dinámico: cambiar de modelo según la complejidad

La lógica es sencilla: si la conversación tiene más de 10 mensajes, se asume que es compleja y se usa el modelo más potente. Si no, se usa el básico.

Esto reduce costes sin sacrificar calidad: el 90% de las conversaciones son cortas y no necesitan el modelo avanzado.

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

In [17]:
# Dos versiones del modelo:
# basic_model → versión lite, rápida y económica
# advanced_model → versión estándar, más potente
basic_model = ChatGoogleGenerativeAI(model=MODEL, temperature=0.9, google_api_key=GOOGLE_API_KEY)
advanced_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.9, google_api_key=GOOGLE_API_KEY)

In [18]:
# Middleware de selección: se ejecuta antes de cada llamada al modelo
# Cuenta los mensajes y decide qué modelo usar
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity"""
    message_count = len(request.state["messages"])

    if message_count > 10:
        model = advanced_model  # Conversación larga → modelo avanzado
    else:
        model = basic_model  # Conversación corta → modelo básico

    return handler(request.override(model=model))

In [19]:
agent = create_agent(
    model=basic_model,
    tools=[],
    middleware=[dynamic_model_selection]
    )

---

## Sección 3 — System prompt + modelo dinámico combinados

Aquí combinamos dos cosas que ya conocemos:
1. El system prompt (personalidad del agente)
2. El modelo dinámico (selección automática según complejidad)

El resultado es un agente especializado en literatura que además optimiza automáticamente qué modelo usa.

In [20]:
# Agente con rol fijo (especialista en literatura) + selección dinámica de modelo
agent = create_agent(
    model=advanced_model,
    tools=[],
    system_prompt = "You are an AI assistant specialized in literarys",
    middleware=[dynamic_model_selection]
    )

In [21]:
# Enviamos un mensaje simple: solo el título de un libro
# El agente, siendo especialista en literatura, elaborará sobre él
result = agent.invoke(
    {"messages": [{"role": "user", "content": "El Quijote"}]}
)

In [13]:
result

{'messages': [HumanMessage(content='El Quijote', additional_kwargs={}, response_metadata={}, id='55bdeee2-b80c-4733-b757-00afcdae3806'),
  AIMessage(content='"El Quijote," or more formally, **"El ingenioso hidalgo Don Quijote de la Mancha"** (The Ingenious Gentleman Don Quixote of La Mancha), is a Spanish novel written by **Miguel de Cervantes Saavedra**. It is widely considered one of the greatest works of literature ever written, and often cited as the first modern novel.\n\nHere\'s a breakdown of what makes it so significant:\n\n**Key Elements and Themes:**\n\n*   **Satire of Chivalric Romances:** The novel is a brilliant parody of the popular chivalric romances of Cervantes\'s time. Don Quixote, an aging gentleman from La Mancha, becomes so engrossed in these stories that he loses his grip on reality and decides to become a knight-errant himself.\n*   **The Idealist vs. The Realist:** The central dynamic of the novel is the contrast between Don Quixote\'s noble, albeit delusional, 

---

## Sección 4 — Stream: respuesta en tiempo real

El streaming muestra la respuesta fragmento a fragmento según se genera, en lugar de esperar a que esté completa. Útil para interfaces de usuario donde el silencio prolongado es mala experiencia.

In [22]:
# Herramienta simulada de tiempo
def get_weather(city: str) -> str:
    """ Get weather for a given city """
    return f"It is always sunny in {city}"

In [23]:
agent = create_agent(
    model = basic_model,
    tools = [get_weather]
)

In [24]:
# Iteramos sobre los fragmentos según llegan
# Cada chunk es un paso del agente (razonar, llamar herramienta, generar respuesta)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Madrid?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 10.297801046s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

---

## Sección 5 — Structured output

Fuerza al modelo a devolver datos con una estructura fija y predecible. Ideal para extracción de información.

In [25]:
from pydantic import BaseModel, Field

In [26]:
# Esquema de datos: define exactamente qué campos debe devolver el modelo
class ConctactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [27]:
agent = create_agent(
    model = basic_model,
    response_format=ConctactInfo  # El agente devolverá siempre un objeto ConctactInfo
)

In [28]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (+34) 666 66 66 66"}]
})

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 7.806132683s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo

In [ ]:
# El resultado estructurado tiene los campos .name, .email, .phone directamente accesibles
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(+34) 666 66 66 66'


---

## Sección 6 — Tools estáticas y dinámicas

### Tools estáticas
El agente siempre tiene las mismas herramientas disponibles.

### Tools dinámicas (pre-registered)
Las herramientas se filtran según el estado de la conversación (autenticación, número de mensajes). Útil para control de acceso.

In [29]:
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Search for information"""
    return f"Results for: {query}"

@tool
def get_weather(city: str) -> str:
    """ Get weather for a given city """
    return f"It is always sunny in {city}"

In [ ]:
# Agente con dos herramientas disponibles permanentemente
agent = create_agent(llm, tools=[search, get_weather])

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# Middleware que filtra herramientas según autenticación
@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Sin autenticación → solo herramientas públicas
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    # Autenticado pero conversación nueva → sin búsqueda avanzada
    elif message_count < 5:
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

# agent = create_agent(
#     model="gpt-4.1",
#     tools=[public_search, private_search, advanced_search],
#     middleware=[state_based_tools]
# )

---

## Sección 7 — Mensajes y chains

Un `ChatPromptTemplate` es una plantilla reutilizable con variables. Al combinarla con el modelo usando el operador `|`, se crea una cadena (chain) que automatiza el flujo: formatear → enviar → recibir.

In [ ]:
# Plantilla de traductor con dos variables: {idioma} y {texto}
chat_prompt = ChatPromptTemplate([
    ("system", "Eres un traductor del español al {idioma} muy preciso"),
    ("human", "{texto}")
])

# Rellenamos las variables y generamos la lista de mensajes
mensajes = chat_prompt.format_messages(texto="Hola mundo", idioma="ruso")

In [ ]:
# Mostramos qué tipo de mensaje es cada uno
for m in mensajes:
    print(f"{type(m)}: {m.content}")

<class 'langchain_core.messages.system.SystemMessage'>: Eres un traductor del español al ruso muy preciso
<class 'langchain_core.messages.human.HumanMessage'>: Hola mundo


In [ ]:
chat = ChatGoogleGenerativeAI(model=MODEL, temperature = 0.5, google_api_key=GOOGLE_API_KEY)

In [ ]:
# La chain conecta: plantilla → modelo
# El operador | (pipe) es la forma de encadenar componentes en LangChain
chain = chat_prompt | chat

In [ ]:
# Ejecutamos la chain con los mensajes ya formateados
resultado = chat.invoke(mensajes)

In [ ]:
resultado

AIMessage(content='Привет, мир', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dd589-b313-7143-8a7b-921048d938bd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 3, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

---

## Sección 8 — RAG: el agente que busca en tus documentos

### ¿Qué es RAG?

**RAG** (Retrieval-Augmented Generation) resuelve uno de los problemas más importantes de los LLMs: **no saben nada de tus documentos privados**.

Un modelo de Gemini fue entrenado con datos de internet hasta cierta fecha. No sabe nada de:
- Los documentos de tu empresa
- Tu base de conocimiento interna
- Los materiales de tu máster

RAG lo soluciona así:
1. Tus documentos se convierten en vectores numéricos (embeddings) y se guardan en una base de datos especial
2. Cuando el usuario hace una pregunta, se buscan los fragmentos más relevantes de esos documentos
3. Esos fragmentos se pasan al LLM como contexto
4. El LLM responde usando esa información real

### ¿Qué es un embedding?

Un embedding es la representación matemática de un texto como una lista de números. Textos similares tienen vectores similares, lo que permite hacer búsquedas por "significado" en lugar de por palabras exactas.

### ¿Qué es un vectorstore?

Es una base de datos diseñada para almacenar y buscar embeddings eficientemente. Aquí usamos **Chroma**, una opción open-source popular.

In [6]:
# Importamos los componentes necesarios para RAG
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # Modelo de embeddings de Google
from langchain_community.vectorstores import Chroma              # Base de datos vectorial
from langchain_classic.schema import Document                    # Objeto que representa un documento

from langchain_huggingface.embeddings import HuggingFaceEmbeddings  # Alternativa gratuita a Google Embeddings

In [7]:
# Creamos los documentos que el agente podrá consultar
# Cada Document tiene:
# - page_content: el texto del documento
# - metadata: información sobre el origen (útil para citar fuentes)
documentos = [
    Document(
        page_content="LangChain es un framework para construir aplicaciones con LLMs. "
                     "Permite encadenar componentes como modelos, herramientas y memoria.",
        metadata={"source": "langchain_intro", "tema": "frameworks"}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) combina búsqueda de documentos "
                     "con generación de texto. Permite al LLM acceder a información externa.",
        metadata={"source": "rag_intro", "tema": "rag"}
    ),
    Document(
        page_content="LangGraph permite crear agentes con estado como grafos dirigidos. "
                     "Es ideal para flujos de trabajo complejos con múltiples pasos.",
        metadata={"source": "langgraph_intro", "tema": "agentes"}
    ),
    Document(
        page_content="Los embeddings son representaciones vectoriales de texto. "
                     "Permiten comparar similitud semántica entre documentos y consultas.",
        metadata={"source": "embeddings_intro", "tema": "embeddings"}
    ),
    Document(
        page_content="AWS Bedrock es un servicio de Amazon para acceder a modelos fundacionales "
                     "de IA generativa a través de una API unificada y segura.",
        metadata={"source": "aws_bedrock", "tema": "cloud"}
    ),
]

In [8]:
import google.generativeai as genai
import os

# 1. Configuración de la API Key
# Asegúrate de que la variable de entorno esté cargada o pásala directamente
genai.configure(api_key=os.getenv("Gemini_API_Key_001"))

# 2. Extracción filtrada
modelos_gemini = [
    m.name.split("/")[-1] 
    for m in genai.list_models() 
    if "generateContent" in m.supported_generation_methods 
    and "gemini" in m.name
]

# 3. Visualización directa
print(f"Modelos detectados: {len(modelos_gemini)}")
for modelo in modelos_gemini:
    print(f" - {modelo}")

Modelos detectados: 24
 - gemini-2.5-flash
 - gemini-2.5-pro
 - gemini-2.0-flash
 - gemini-2.0-flash-001
 - gemini-2.0-flash-lite-001
 - gemini-2.0-flash-lite
 - gemini-2.5-flash-preview-tts
 - gemini-2.5-pro-preview-tts
 - gemini-flash-latest
 - gemini-flash-lite-latest
 - gemini-pro-latest
 - gemini-2.5-flash-lite
 - gemini-2.5-flash-image
 - gemini-3-pro-preview
 - gemini-3-flash-preview
 - gemini-3.1-pro-preview
 - gemini-3.1-pro-preview-customtools
 - gemini-3.1-flash-lite-preview
 - gemini-3-pro-image-preview
 - gemini-3.1-flash-image-preview
 - gemini-3.1-flash-tts-preview
 - gemini-robotics-er-1.5-preview
 - gemini-robotics-er-1.6-preview
 - gemini-2.5-computer-use-preview-10-2025


In [9]:
# Necesitamos la clave de API también para los embeddings
import os
from dotenv import load_dotenv

# Lee el archivo .env si existe
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# En Google Colab usa: from google.colab import userdata; GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
# En local con .env usa: GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
print("✅ API Key cargada")

✅ API Key cargada


In [10]:
import google.generativeai as genai
import os

# Configuración
genai.configure(api_key=os.getenv("Gemini_API_Key_001"))

# Extracción de modelos de embeddings
modelos_embeddings = [
    m.name.split("/")[-1] 
    for m in genai.list_models() 
    if "embedContent" in m.supported_generation_methods
]

# Mostrar resultados
print(f"Modelos de Embeddings detectados: {len(modelos_embeddings)}")
for modelo in modelos_embeddings:
    print(f" - {modelo}")

Modelos de Embeddings detectados: 3
 - gemini-embedding-001
 - gemini-embedding-2-preview
 - gemini-embedding-2


In [11]:
# Especificamos el modelo de embeddings de Google que usaremos
modelo_embedding = "gemini-embedding-001"

In [12]:
# Creamos el modelo de embeddings
# Este modelo convierte cada fragmento de texto en un vector de números
embeddings = GoogleGenerativeAIEmbeddings(model=modelo_embedding, google_api_key=GOOGLE_API_KEY)

# ALTERNATIVA GRATUITA (sin API Key): HuggingFace
# embeddings_hugging = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

In [13]:

print(f"Documentos a procesar: {len(documentos)}")
# Prueba un embedding rápido
test_vector = embeddings.embed_query("Hola mundo")
print(f"Dimensión del vector: {len(test_vector)}")

Documentos a procesar: 5
Dimensión del vector: 3072


In [14]:
# Creamos la base de datos vectorial (vectorstore)
# Chroma.from_documents() hace tres cosas automáticamente:
# 1. Toma cada documento
# 2. Lo convierte en un vector usando el modelo de embeddings
# 3. Guarda el vector + el texto original en la base de datos
vectorstore = Chroma.from_documents(
    documents = documentos,
    embedding = embeddings,
    collection_name="master_ia_doc"  # Nombre de la colección en Chroma
)

In [16]:
from langchain.tools import tool

In [17]:
# El retriever es el objeto que busca en el vectorstore
# search_kwargs={"k": 2} → devuelve los 2 fragmentos más relevantes para cada consulta
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [18]:
# Convertimos el retriever en una herramienta que el agente puede usar
# La descripción le dice al agente cuándo debe usar esta herramienta
@tool
def buscar_documentacion(query: str) -> str:
    """ Busca información relevante en la documentacion del master de IA.
        Usala cuando necesites informacion sobre LangChain, RAG, LangGraph, embeddings, AWS BedRock
    """
    # Busca en el vectorstore los 2 fragmentos más similares a la consulta
    docs = retriever.invoke(query)
    if not docs:
        return "No encontré información relevante sobre ese tema"

    resultados = []

    # Formateamos los resultados incluyendo la fuente para trazabilidad
    for i, doc in enumerate(docs, 1):
        fuente = doc.metadata.get('source', 'desconocida')
        resultados.append(f"[Fuente: {fuente}] \n {doc.page_content}")

    return "\n\n".join(resultados)

In [19]:
# Probamos la herramienta directamente antes de integrarla en el agente
# .invoke() la llama como si fuera el agente quien la llama
resultado = buscar_documentacion.invoke({"query": "¿Que es el RAG?"})

In [20]:
# Vemos qué devuelve la herramienta: los fragmentos más relevantes encontrados
resultado

'[Fuente: rag_intro] \n RAG (Retrieval-Augmented Generation) combina búsqueda de documentos con generación de texto. Permite al LLM acceder a información externa.\n\n[Fuente: langchain_intro] \n LangChain es un framework para construir aplicaciones con LLMs. Permite encadenar componentes como modelos, herramientas y memoria.'

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [63]:
# Para el agente RAG usamos el modelo más potente (temperature=0 para respuestas factuales)
llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0,  # 0 = máxima precisión, sin variación
    google_api_key = GOOGLE_API_KEY
)

In [64]:
# System prompt que instruye al agente a SIEMPRE buscar antes de responder sobre temas técnicos
system_prompt = """Eres un asistente experto en IA Generativa para un máster universitario.
Tienes acceso a documentación del máster. Cuando el usuario pregunta algo técnico,
SIEMPRE usa la herramienta `buscar_documentacion` para fundamentar tu respuesta.
Si la pregunta es de conversación general, puedes responder directamente.
Responde siempre en español."""

In [65]:
# Creamos el agente RAG: combina el LLM con la herramienta de búsqueda
agente_rag = create_agent(
    model = llm,
    tools=[buscar_documentacion],  # El agente puede buscar en los documentos del máster
    system_prompt=system_prompt
)

In [66]:
from langchain_core.messages import HumanMessage

In [67]:
# Función helper que hace una pregunta al agente y muestra el proceso paso a paso
# Nos permite ver: qué herramientas usó, qué recuperó, y cuál fue la respuesta final
def preguntar_al_agente(pregunta: str):
    """Función helper para hacer preguntas al agente y ver el proceso."""
    print(f"\n{'='*60}")
    print(f"👤 Pregunta: {pregunta}")
    print('='*60)

    respuesta = agente_rag.invoke({
        "messages": [HumanMessage(content=pregunta)]
    })

    # Mostramos el proceso paso a paso
    print("\n🔄 Proceso del agente:")
    for msg in respuesta["messages"]:
        tipo = msg.__class__.__name__
        if tipo == "HumanMessage":
            print(f"  👤 Usuario: {msg.content[:100]}")
        elif tipo == "AIMessage":
            if msg.tool_calls:
                # El agente decidió usar una herramienta
                for tc in msg.tool_calls:
                    print(f"  🔧 Agente decide usar herramienta: {tc['name']}")
                    print(f"     Query: {tc['args'].get('query', '')}")
            else:
                # El agente está generando la respuesta final
                print(f"\n  🤖 Respuesta final:\n  {msg.content}")
        elif tipo == "ToolMessage":
            # Resultado que devolvió la herramienta (los documentos recuperados)
            print(f"  📚 Resultado recuperado: {msg.content[:150]}...")

In [ ]:
# Prueba: preguntamos sobre un tema que está en los documentos del máster
# El agente buscará, encontrará nada relevante, y responderá con la ayuda de los documentos incorporados
preguntar_al_agente("Que es LangGraph y para que sirve?")


👤 Pregunta: Que es LangGraph y para que sirve?

🔄 Proceso del agente:
  👤 Usuario: Que es LangGraph y para que sirve?
  🔧 Agente decide usar herramienta: buscar_documentacion
     Query: LangGraph qué es y para qué sirve
  📚 Resultado recuperado: [Fuente: langgraph_intro] 
 LangGraph permite crear agentes con estado como grafos dirigidos. Es ideal para flujos de trabajo complejos con múltiples ...
  🔧 Agente decide usar herramienta: buscar_documentacion
     Query: componentes LangGraph nodos bordes estado
  📚 Resultado recuperado: [Fuente: langgraph_intro] 
 LangGraph permite crear agentes con estado como grafos dirigidos. Es ideal para flujos de trabajo complejos con múltiples ...

  🤖 Respuesta final:
  [{'type': 'text', 'text': 'LangGraph es una biblioteca construida sobre LangChain que se utiliza para crear aplicaciones de agentes con **estado** (stateful) y flujos de trabajo complejos representados como **grafos dirigidos**.\n\nSegún la documentación del máster, sus característi

In [ ]:
# Prueba: preguntamos sobre un tema que NO está en los documentos del máster
# El agente buscará, no encontrará nada relevante, y responderá con su conocimiento general
preguntar_al_agente("Que ingredientes tiene la tortilla de patata?")


👤 Pregunta: Que ingredientes tiene la tortilla de patata?

🔄 Proceso del agente:
  👤 Usuario: Que ingredientes tiene la tortilla de patata?

  🤖 Respuesta final:
  [{'type': 'text', 'text': 'La tortilla de patata clásica es un plato sencillo pero delicioso. Los ingredientes básicos son:\n\n1.  **Patatas:** Preferiblemente de una variedad que no se deshaga mucho al freír.\n2.  **Huevos:** Frescos y bien batidos.\n3.  **Aceite de oliva:** Para freír las patatas (y la cebolla si decides ponerle).\n4.  **Sal:** Al gusto.\n5.  **Cebolla (opcional):** Este es el ingrediente de la eterna discordia en España; hay quienes la consideran imprescindible y quienes prefieren la tortilla sin ella.\n\n**Preparación básica:** Se cortan las patatas en láminas finas y se fríen a fuego lento en abundante aceite hasta que estén tiernas. Se escurren, se mezclan con el huevo batido y se cuaja la mezcla en una sartén con un poquito de aceite por ambos lados. ¡Buen provecho!', 'extras': {'signature': 'EswFCsk

---

## Sección 9 — Memoria: el agente que recuerda la conversación

### El problema sin memoria

Por defecto, cada vez que llamas a `agent.invoke()`, el agente empieza desde cero. No recuerda lo que dijiste antes. Es como llamar al servicio de atención al cliente y que cada respuesta te atienda alguien diferente que no sabe nada de tu caso.

El código de abajo demuestra este problema: después de presentarte, si preguntas cómo te llamas, el agente no lo sabe.

In [ ]:
import google.generativeai as genai
import os

# 1. Configuración de la API Key
# Asegúrate de que la variable de entorno esté cargada o pásala directamente
genai.configure(api_key=os.getenv("Gemini_API_Key_001"))

# 2. Extracción filtrada
modelos_gemini = [
    m.name.split("/")[-1] 
    for m in genai.list_models() 
    if "generateContent" in m.supported_generation_methods 
    and "gemini" in m.name
]

# 3. Visualización directa
print(f"Modelos detectados: {len(modelos_gemini)}")
for modelo in modelos_gemini:
    print(f" - {modelo}")

Modelos detectados: 24
 - gemini-2.5-flash
 - gemini-2.5-pro
 - gemini-2.0-flash
 - gemini-2.0-flash-001
 - gemini-2.0-flash-lite-001
 - gemini-2.0-flash-lite
 - gemini-2.5-flash-preview-tts
 - gemini-2.5-pro-preview-tts
 - gemini-flash-latest
 - gemini-flash-lite-latest
 - gemini-pro-latest
 - gemini-2.5-flash-lite
 - gemini-2.5-flash-image
 - gemini-3-pro-preview
 - gemini-3-flash-preview
 - gemini-3.1-pro-preview
 - gemini-3.1-pro-preview-customtools
 - gemini-3.1-flash-lite-preview
 - gemini-3-pro-image-preview
 - gemini-3.1-flash-image-preview
 - gemini-3.1-flash-tts-preview
 - gemini-robotics-er-1.5-preview
 - gemini-robotics-er-1.6-preview
 - gemini-2.5-computer-use-preview-10-2025


In [57]:
# Creamos un modelo con temperatura 0 (respuestas precisas y consistentes)
llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0, google_api_key=GOOGLE_API_KEY)

In [58]:
# Agente SIN memoria — cada llamada es independiente
agente_sin_memoria = create_agent(
    model=llm,
    tools=[]
)

In [59]:
# Primer mensaje: el usuario se presenta
r1 = agente_sin_memoria.invoke({"messages":[HumanMessage("Hola, me llamo Carlos y soy del master de IA")]})

In [60]:
# El agente responde a la presentación
r1

{'messages': [HumanMessage(content='Hola, me llamo Carlos y soy del master de IA', additional_kwargs={}, response_metadata={}, id='3ddd6bd4-59a3-42d1-8181-55ccfd9e8da5'),
  AIMessage(content=[{'type': 'text', 'text': '¡Hola, Carlos! Mucho gusto. Es un placer saludarte.\n\nQué bien que estés cursando un **Máster en IA**. Es un campo fascinante y que está evolucionando a una velocidad increíble.\n\n¿En qué etapa del máster te encuentras? Si necesitas ayuda con algún concepto teórico (como redes neuronales, transformers, refuerzo...), apoyo con código (Python, PyTorch, TensorFlow) o simplemente quieres debatir sobre las últimas novedades del sector, ¡aquí me tienes!\n\n¿Hay algún tema en particular que estés estudiando ahora mismo o algún proyecto que tengas entre manos?', 'extras': {'signature': 'EskLCsYLAQw51scmnRCUVmJvMfda2h/+uY3eVUu0e/jMKiTk1h38kBlStKSmpeg2g2ln0CvPyDT7B2O8w170HB9QaBPjwWo3K26rVa4uSLFzZw98lxcDnEQ5jFncvqKb/8NxQFSVtCtG9ZwMrL2OB0RUsad712n7Q8kqv9LiWmUNg08mR7rPM64IXd1B3BxL7z

In [61]:
# Segunda llamada: preguntamos el nombre
# PROBLEMA: esta llamada no tiene ninguna relación con la anterior
# El agente no ha recibido el mensaje anterior, no sabe quién es Carlos
r2 = agente_sin_memoria.invoke({"messages":[HumanMessage("¿Como me llamo?")]})

In [62]:
# El agente responde que no sabe el nombre → confirma el problema de falta de memoria
r2

{'messages': [HumanMessage(content='¿Como me llamo?', additional_kwargs={}, response_metadata={}, id='9685e35d-3939-492e-89e7-cfcd24d759a9'),
  AIMessage(content=[{'type': 'text', 'text': 'No lo sé, ya que soy una inteligencia artificial y no tengo acceso a tu información personal ni a tu identidad real a menos que tú me lo digas.\n\n¿Cómo te llamas? Si me lo dices, lo recordaré durante nuestra conversación.', 'extras': {'signature': 'ErQJCrEJAQw51sf7yFX/UP4OeVduDGWv+8DJHlTUwm920jdaN6Eksa8SaFyRqeoMDziUL7jP8EAY7fvbOFc3dxtr4ocua/mqZB27z5U+BvTcQ77fcRiFHZilO8Rs97rI/Isotg8idgqh2uXDZttttInPlnXMIwH/OEGHRVPB9b/5sUk4E0Z5njSyZw0Rbm61zb6H1VczPcbM5js6LaUGrWx5BSF/btPNvou7SsbpMxm0TXLaez5GpFSPgPq+PWkT+pxt9MEWSZKht4n2J49PhpyVIz9LIRZDURrSowVuyeb9a66H5RmX3GJ7qo/nX8TMnoyNnn+OkC7VbTNcQo1l1Y3pvpfRvtq0vioePXd6VzS63456Etqxdb17hz9SAj+JGBY2dF+j7h5wpHMax7GBc5LRvkB2dDX5tjaKub49TH14p6N1RGmAorz/U1YcZ5k2kID5VZrsdmDpxDV6aMU+yjLCfG8iSP0Ki9V7eHBqEkk5fbI3AcnNhVrXWSoaMEij6d5IObFhMts9Lu8g7rP/mIyIUolabb8UP2cOi/HW0uWOedNEA

### La solución: InMemorySaver

**InMemorySaver** es un "checkpointer" (guardián de estado) que:
1. Guarda automáticamente todos los mensajes de una conversación
2. Los identifica por un `thread_id` (un identificador único de conversación)
3. Los recupera y los añade al contexto en la siguiente llamada

Con esto, el agente "recuerda" porque todos los mensajes anteriores se incluyen en cada nueva llamada.

El `thread_id` es clave: permite tener múltiples conversaciones independientes en paralelo (diferentes usuarios, diferentes sesiones).

In [70]:
# Importamos InMemorySaver: guarda el estado de la conversación en memoria RAM
# (se pierde al reiniciar el kernel, pero es suficiente para una sesión)
from langgraph.checkpoint.memory import InMemorySaver

In [71]:
# Creamos el guardián de estado — un único objeto para todos los threads
checkpointer = InMemorySaver()

In [72]:
# Agente CON memoria: añadimos el checkpointer
# La única diferencia con el agente sin memoria es checkpointer=checkpointer
agente_con_memoria = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer  # ← Esto activa la memoria
)

In [73]:
# El thread_id identifica esta conversación específica
# Todas las llamadas con el mismo thread_id comparten memoria
config_thread1 = {"configurable": {"thread_id": "conversacion-1"}}

In [74]:
# Primer mensaje: el usuario se presenta
# config=config_thread1 → asocia este mensaje al thread "conversacion-1"
r1 = agente_con_memoria.invoke(
    {"messages":[HumanMessage("Hola, me llamo Carlos y soy del master de IA")]},
    config=config_thread1
    )

In [77]:
r1

{'messages': [HumanMessage(content='Hola, me llamo Carlos y soy del master de IA', additional_kwargs={}, response_metadata={}, id='79d9596f-40b5-49b6-80f4-b62590014b6f'),
  AIMessage(content=[{'type': 'text', 'text': '¡Hola, Carlos! Mucho gusto. Es un placer saludarte.\n\nQué bien que estés cursando un **Máster en IA**. Es un campo fascinante y que está evolucionando a una velocidad increíble.\n\n¿En qué etapa del máster te encuentras? Si necesitas ayuda con algún concepto teórico (como redes neuronales, transformers, refuerzo...), apoyo con código (Python, PyTorch, TensorFlow) o simplemente quieres debatir sobre las últimas novedades del sector, ¡aquí me tienes!\n\n¿Hay algún tema en particular que estés estudiando ahora mismo o algún proyecto que tengas entre manos?', 'extras': {'signature': 'EskLCsYLAQw51sfzWVcqOm8f/x7GPA1MQIIHWyisjdG8qESS6gA7svAE8ydjJG+JPKdco4Pjiw3EWKIA/cYJJZfOkOfXug597wIh35u95zZwjyiznblMUcS4DnQqCxN33AybpAm7YCGCg9Glc/hgoLW4dC3BwtxauEV8qxx6MI81JN23mW144mA4ZZHTbtYgAS

In [75]:
# Segundo mensaje: preguntamos el nombre
# Como usamos el mismo thread_id, el agente tiene acceso al mensaje anterior
# Ahora sí sabe que Carlos se llama Carlos y estudia el máster de IA
r2 = agente_con_memoria.invoke(
    {"messages":[HumanMessage("¿Como me llamo y que estudio?")]},
    config=config_thread1  # ← Mismo thread = misma conversación con memoria
    )

In [76]:
# El agente ahora responde correctamente: "Te llamas Carlos y estudias el máster de IA"
r2

{'messages': [HumanMessage(content='Hola, me llamo Carlos y soy del master de IA', additional_kwargs={}, response_metadata={}, id='79d9596f-40b5-49b6-80f4-b62590014b6f'),
  AIMessage(content=[{'type': 'text', 'text': '¡Hola, Carlos! Mucho gusto. Es un placer saludarte.\n\nQué bien que estés cursando un **Máster en IA**. Es un campo fascinante y que está evolucionando a una velocidad increíble.\n\n¿En qué etapa del máster te encuentras? Si necesitas ayuda con algún concepto teórico (como redes neuronales, transformers, refuerzo...), apoyo con código (Python, PyTorch, TensorFlow) o simplemente quieres debatir sobre las últimas novedades del sector, ¡aquí me tienes!\n\n¿Hay algún tema en particular que estés estudiando ahora mismo o algún proyecto que tengas entre manos?', 'extras': {'signature': 'EskLCsYLAQw51sfzWVcqOm8f/x7GPA1MQIIHWyisjdG8qESS6gA7svAE8ydjJG+JPKdco4Pjiw3EWKIA/cYJJZfOkOfXug597wIh35u95zZwjyiznblMUcS4DnQqCxN33AybpAm7YCGCg9Glc/hgoLW4dC3BwtxauEV8qxx6MI81JN23mW144mA4ZZHTbtYgAS

---

## Resumen del notebook

| Capacidad | Lo que aporta | Componente clave |
|-----------|---------------|------------------|
| Modelo estático | Simplicidad | `create_agent(llm, tools=None)` |
| Modelo dinámico | Optimización de coste | `@wrap_model_call` middleware |
| System prompt | Personalidad y rol fijo | `system_prompt="..."` |
| Stream | Experiencia de usuario fluida | `agent.stream()` |
| Structured output | Datos procesables en código | `response_format=MiClase` |
| Tools | Acciones reales | `@tool` + `tools=[...]` |
| **RAG** | **Consultar documentos propios** | **Chroma + embeddings + retriever** |
| **Memoria** | **Recordar la conversación** | **InMemorySaver + thread_id** |